# 🎨 Embeddings & Semantic Similarity Visualization

**Estimated time: 12-15 minutes**

## What are Embeddings?

Embeddings are numerical representations of text (words, sentences, or documents) in a high-dimensional vector space. They capture **semantic meaning** - words or sentences with similar meanings are positioned close to each other in this space.

Think of it like a map where:
- Each word/sentence is a point
- Similar concepts cluster together
- Distance measures similarity

### Why are they useful?

- 🔍 **Semantic Search**: Find similar documents based on meaning, not just keywords
- 📊 **Clustering**: Group similar items together
- 🤖 **Transfer Learning**: Use pre-trained models for downstream tasks
- 🎯 **Recommendation Systems**: Find similar products, articles, or content

### How do we measure similarity?

**Cosine Similarity** measures the angle between two vectors:
- Score of 1.0 = identical meaning
- Score of 0.0 = unrelated
- Score of -1.0 = opposite (rare in practice)

Let's dive in! 🚀

## 📦 Setup & Installation

In [11]:
# Import libraries
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.manifold import TSNE
import pandas as pd
import umap
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


## 🤖 Load Embedding Model

We'll use **all-MiniLM-L6-v2**, a compact but powerful model that:
- Generates 384-dimensional embeddings
- Is fast and efficient
- Works great for semantic similarity tasks

In [2]:
# Load the model (this may take a minute on first run)
model = SentenceTransformer('all-MiniLM-L6-v2')

print(f"✅ Model loaded!")
print(f"   Embedding dimension: {model.get_sentence_embedding_dimension()}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded!
   Embedding dimension: 384


## 🔢 Example 1: Word Embeddings & Similarity

Let's start simple by comparing individual words.

In [3]:
# Define some words
words = ['king', 'queen', 'man', 'woman', 'prince', 'princess', 
         'dog', 'cat', 'puppy', 'kitten',
         'car', 'vehicle', 'bicycle', 'motorcycle']

# Generate embeddings
word_embeddings = model.encode(words)

print(f"Generated {len(word_embeddings)} embeddings")
print(f"Each embedding has {len(word_embeddings[0])} dimensions")
print(f"\nExample - 'king' embedding (first 10 values):")
print(word_embeddings[0][:10])

Generated 14 embeddings
Each embedding has 384 dimensions

Example - 'king' embedding (first 10 values):
[-0.05959932  0.05051232 -0.06951012  0.07968019 -0.04674765  0.00098894
  0.07904322 -0.01273938  0.05839579 -0.03140246]


In [4]:
# Calculate similarity between all words
similarity_matrix = cosine_similarity(word_embeddings)

# Create an interactive heatmap
fig = go.Figure(data=go.Heatmap(
    z=similarity_matrix,
    x=words,
    y=words,
    colorscale='RdYlGn',
    text=np.round(similarity_matrix, 2),
    texttemplate='%{text}',
    textfont={"size": 10},
    hoverongaps=False,
    colorbar=dict(title="Similarity")
))

fig.update_layout(
    title='Word Similarity Heatmap (Cosine Similarity)',
    xaxis_title='Words',
    yaxis_title='Words',
    width=800,
    height=700
)

fig.show()

print("\n💡 Observations:")
print("   - Diagonal is 1.0 (words are identical to themselves)")
print("   - 'king' and 'queen' are very similar")
print("   - 'dog' and 'cat' are similar (both pets)")
print("   - 'king' and 'dog' are less similar (different concepts)")


💡 Observations:
   - Diagonal is 1.0 (words are identical to themselves)
   - 'king' and 'queen' are very similar
   - 'dog' and 'cat' are similar (both pets)
   - 'king' and 'dog' are less similar (different concepts)


## 🧮 Example 2: Semantic Arithmetic (Vector Math)

One of the coolest properties of embeddings: we can do **math** with meaning!

Famous example: `king - man + woman ≈ queen`

Let's test this!

In [5]:
def find_most_similar(target_embedding, word_embeddings, words, exclude_words=None, top_k=5):
    """Find the most similar words to a target embedding"""
    similarities = cosine_similarity([target_embedding], word_embeddings)[0]
    
    # Create list of (word, similarity) pairs
    word_similarities = list(zip(words, similarities))
    
    # Filter out excluded words
    if exclude_words:
        word_similarities = [(w, s) for w, s in word_similarities if w not in exclude_words]
    
    # Sort by similarity (descending)
    word_similarities.sort(key=lambda x: x[1], reverse=True)
    
    return word_similarities[:top_k]

# Get individual embeddings
king_emb = model.encode('king')
man_emb = model.encode('man')
woman_emb = model.encode('woman')

# Perform vector arithmetic
result_emb = king_emb - man_emb + woman_emb

# Find what this new vector is closest to
candidates = ['queen', 'princess', 'king', 'prince', 'lady', 'monarch', 'duchess', 'empress']
candidate_embeddings = model.encode(candidates)

similar_words = find_most_similar(result_emb, candidate_embeddings, candidates, top_k=5)

print("🎯 Vector Arithmetic: king - man + woman = ?\n")
print("Most similar words:")
for i, (word, score) in enumerate(similar_words, 1):
    bar = '█' * int(score * 50)
    print(f"{i}. {word:15s} {bar} {score:.4f}")

print("\n✅ It works! The model understands the gender relationship!")

🎯 Vector Arithmetic: king - man + woman = ?

Most similar words:
1. king            ███████████████████████████████ 0.6306
2. queen           ████████████████████████████ 0.5795
3. monarch         ███████████████████████████ 0.5469
4. lady            ████████████████████████ 0.4849
5. princess        ██████████████████████ 0.4418

✅ It works! The model understands the gender relationship!


In [6]:
# Let's try more examples!
def semantic_analogy(word1, word2, word3, candidates):
    """Solve: word1 is to word2 as word3 is to ?"""
    emb1 = model.encode(word1)
    emb2 = model.encode(word2)
    emb3 = model.encode(word3)
    
    # word1 - word2 + word3
    result = emb1 - emb2 + emb3
    
    candidate_embeddings = model.encode(candidates)
    similar = find_most_similar(result, candidate_embeddings, candidates, 
                                exclude_words=[word1, word2, word3], top_k=3)
    
    print(f"\n{word1} - {word2} + {word3} = ?")
    print(f"Top predictions:")
    for word, score in similar:
        print(f"  → {word} ({score:.3f})")

# Test various analogies
print("🧪 Semantic Analogies Test:\n" + "="*50)

semantic_analogy('Paris', 'France', 'London', 
                ['England', 'Britain', 'UK', 'Germany', 'Italy', 'Spain'])

semantic_analogy('doctor', 'hospital', 'teacher', 
                ['school', 'university', 'classroom', 'library', 'office', 'laboratory'])

semantic_analogy('puppy', 'dog', 'kitten', 
                ['cat', 'feline', 'tiger', 'lion', 'pet', 'animal'])

🧪 Semantic Analogies Test:

Paris - France + London = ?
Top predictions:
  → UK (0.476)
  → Britain (0.351)
  → England (0.235)

doctor - hospital + teacher = ?
Top predictions:
  → classroom (0.444)
  → school (0.419)
  → office (0.292)

puppy - dog + kitten = ?
Top predictions:
  → cat (0.525)
  → feline (0.489)
  → tiger (0.414)


## 🔍 Example 3: Semantic Search

Now let's do something practical: **search research papers by semantic meaning**!

We'll create a mini research paper database and find similar papers based on their titles.

In [7]:
# Sample research paper titles (diverse topics)
papers = [
    "Deep Learning for Image Classification Using Convolutional Neural Networks",
    "Neural Networks for Computer Vision: A Comprehensive Review",
    "Attention Mechanisms in Natural Language Processing",
    "BERT: Pre-training of Deep Bidirectional Transformers",
    "Quantum Computing: Algorithms and Applications",
    "Quantum Error Correction and Fault-Tolerant Computing",
    "Climate Change Impact on Marine Ecosystems",
    "Ocean Acidification and Coral Reef Degradation",
    "Machine Learning in Healthcare: Predictive Analytics",
    "AI-Based Disease Diagnosis Using Medical Imaging",
    "Blockchain Technology for Secure Financial Transactions",
    "Cryptocurrency Mining and Environmental Sustainability",
    "Gene Editing with CRISPR-Cas9: Ethical Implications",
    "Synthetic Biology and Genetic Engineering Advances",
    "Renewable Energy Storage Solutions for Smart Grids",
    "Solar Panel Efficiency and Energy Conversion Optimization"
]

# Generate embeddings for all papers
paper_embeddings = model.encode(papers)

print(f"✅ Indexed {len(papers)} research papers\n")

✅ Indexed 16 research papers



In [8]:
def semantic_search(query, papers, paper_embeddings, top_k=5):
    """Search for papers similar to the query"""
    # Encode the query
    query_embedding = model.encode(query)
    
    # Calculate similarities
    similarities = cosine_similarity([query_embedding], paper_embeddings)[0]
    
    # Get top results
    top_indices = np.argsort(similarities)[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        results.append({
            'paper': papers[idx],
            'similarity': similarities[idx]
        })
    
    return results

# Test semantic search
queries = [
    "transformers for text understanding",
    "environmental impact of oceans",
    "medical AI applications"
]

for query in queries:
    print(f"\n🔍 Query: '{query}'")
    print("="*80)
    
    results = semantic_search(query, papers, paper_embeddings, top_k=3)
    
    for i, result in enumerate(results, 1):
        score_bar = '█' * int(result['similarity'] * 40)
        print(f"\n{i}. [Score: {result['similarity']:.3f}] {score_bar}")
        print(f"   {result['paper']}")


🔍 Query: 'transformers for text understanding'

1. [Score: 0.466] ██████████████████
   BERT: Pre-training of Deep Bidirectional Transformers

2. [Score: 0.316] ████████████
   Attention Mechanisms in Natural Language Processing

3. [Score: 0.146] █████
   Deep Learning for Image Classification Using Convolutional Neural Networks

🔍 Query: 'environmental impact of oceans'

1. [Score: 0.668] ██████████████████████████
   Climate Change Impact on Marine Ecosystems

2. [Score: 0.504] ████████████████████
   Ocean Acidification and Coral Reef Degradation

3. [Score: 0.240] █████████
   Cryptocurrency Mining and Environmental Sustainability

🔍 Query: 'medical AI applications'

1. [Score: 0.637] █████████████████████████
   AI-Based Disease Diagnosis Using Medical Imaging

2. [Score: 0.458] ██████████████████
   Machine Learning in Healthcare: Predictive Analytics

3. [Score: 0.235] █████████
   Deep Learning for Image Classification Using Convolutional Neural Networks


In [9]:
# Interactive search widget
from IPython.display import display, HTML

def interactive_search(user_query):
    results = semantic_search(user_query, papers, paper_embeddings, top_k=5)
    
    html = f"<h3 style='color: #2E86AB;'>🔍 Search Results for: '{user_query}'</h3>"
    
    for i, result in enumerate(results, 1):
        score_pct = int(result['similarity'] * 100)
        color = '#4CAF50' if score_pct > 50 else '#FF9800' if score_pct > 30 else '#F44336'
        
        html += f"""
        <div style='margin: 15px 0; padding: 15px; background-color: #f5f5f5; border-left: 4px solid {color}; border-radius: 4px;'>
            <div style='font-weight: bold; color: {color};'>#{i} - Similarity: {score_pct}%</div>
            <div style='margin-top: 8px; color: #333;'>{result['paper']}</div>
        </div>
        """
    
    display(HTML(html))

# Try it out!
print("Try searching for papers! Examples:")
print("- 'quantum algorithms'")
print("- 'green energy solutions'")
print("- 'gene modification techniques'\n")

interactive_search("quantum algorithms")

Try searching for papers! Examples:
- 'quantum algorithms'
- 'green energy solutions'
- 'gene modification techniques'



## 📊 Example 4: 2D Visualization with t-SNE

Embeddings live in high-dimensional space (384 dimensions in our case). Let's compress them to 2D so we can visualize them!

**t-SNE** (t-Distributed Stochastic Neighbor Embedding) preserves local structure - similar items stay close together.

In [12]:
# Create categories for coloring
categories = [
    'AI/ML', 'AI/ML',  # Deep learning papers
    'NLP', 'NLP',      # Language papers
    'Quantum', 'Quantum',  # Quantum papers
    'Environment', 'Environment',  # Climate papers
    'Healthcare', 'Healthcare',  # Medical papers
    'Blockchain', 'Blockchain',  # Crypto papers
    'Biology', 'Biology',  # Gene editing papers
    'Energy', 'Energy'  # Renewable energy papers
]

# Apply t-SNE
print("Running t-SNE dimensionality reduction...")
tsne = TSNE(n_components=2, random_state=42, perplexity=5)
embeddings_2d = tsne.fit_transform(paper_embeddings)

# Create DataFrame for plotting
df = pd.DataFrame({
    'x': embeddings_2d[:, 0],
    'y': embeddings_2d[:, 1],
    'paper': papers,
    'category': categories
})

# Create interactive scatter plot
fig = px.scatter(
    df, 
    x='x', 
    y='y', 
    color='category',
    hover_data={'paper': True, 'x': False, 'y': False, 'category': True},
    title='Research Papers in 2D Space (t-SNE)',
    width=900,
    height=700,
    color_discrete_sequence=px.colors.qualitative.Set2
)

fig.update_traces(
    marker=dict(size=15, line=dict(width=2, color='white')),
    selector=dict(mode='markers')
)

fig.update_layout(
    xaxis_title='t-SNE Dimension 1',
    yaxis_title='t-SNE Dimension 2',
    legend_title='Research Area',
    hovermode='closest'
)

fig.show()

print("\n💡 Notice how papers on similar topics cluster together!")
print("   Hover over points to see the paper titles.")

Running t-SNE dimensionality reduction...



💡 Notice how papers on similar topics cluster together!
   Hover over points to see the paper titles.


## 🗺️ Example 5: UMAP Visualization (Alternative to t-SNE)

**UMAP** (Uniform Manifold Approximation and Projection) is another dimensionality reduction technique that often preserves both local and global structure better than t-SNE.

In [ ]:
# Apply UMAP
print("Running UMAP dimensionality reduction...")
reducer = umap.UMAP(n_neighbors=5, min_dist=0.3, random_state=42)
embeddings_umap = reducer.fit_transform(paper_embeddings)

# Create DataFrame
df_umap = pd.DataFrame({
    'x': embeddings_umap[:, 0],
    'y': embeddings_umap[:, 1],
    'paper': papers,
    'category': categories
})

# Create plot
fig = px.scatter(
    df_umap, 
    x='x', 
    y='y', 
    color='category',
    hover_data={'paper': True, 'x': False, 'y': False, 'category': True},
    title='Research Papers in 2D Space (UMAP)',
    width=900,
    height=700,
    color_discrete_sequence=px.colors.qualitative.Bold
)

fig.update_traces(
    marker=dict(size=15, line=dict(width=2, color='white')),
    selector=dict(mode='markers')
)

fig.update_layout(
    xaxis_title='UMAP Dimension 1',
    yaxis_title='UMAP Dimension 2',
    legend_title='Research Area',
    hovermode='closest'
)

fig.show()

print("\n💡 UMAP often creates tighter, more separated clusters than t-SNE!")

## 🎯 Example 6: Sentence Similarity Comparison

Let's see how embeddings handle sentence-level semantics.

In [ ]:
# Example sentences
sentences = [
    "The cat sits on the mat.",
    "A feline rests on the rug.",
    "Dogs are playing in the park.",
    "The weather is beautiful today.",
    "It's a sunny and pleasant day.",
    "Machine learning models require large datasets.",
    "AI systems need lots of training data.",
    "Python is a programming language.",
]

sentence_embeddings = model.encode(sentences)
sentence_similarities = cosine_similarity(sentence_embeddings)

# Create heatmap
fig = go.Figure(data=go.Heatmap(
    z=sentence_similarities,
    x=[f"S{i+1}" for i in range(len(sentences))],
    y=[f"S{i+1}" for i in range(len(sentences))],
    colorscale='Viridis',
    text=np.round(sentence_similarities, 2),
    texttemplate='%{text}',
    textfont={"size": 9},
    colorbar=dict(title="Similarity")
))

fig.update_layout(
    title='Sentence Similarity Matrix',
    width=700,
    height=700
)

fig.show()

# Print sentences for reference
print("\nSentence Reference:")
print("="*80)
for i, sent in enumerate(sentences, 1):
    print(f"S{i}: {sent}")

print("\n💡 Key observations:")
print("   - S1 and S2 are very similar (same meaning, different words)")
print("   - S4 and S5 are similar (both about weather)")
print("   - S6 and S7 are similar (both about ML/AI data)")
print("   - Different topics have low similarity")

## 📈 Example 7: Clustering Visualization

Let's automatically cluster our research papers and visualize them!

In [ ]:
from sklearn.cluster import KMeans

# Perform K-means clustering
n_clusters = 5
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
clusters = kmeans.fit_predict(paper_embeddings)

# Add cluster labels to DataFrame
df_umap['cluster'] = [f'Cluster {c+1}' for c in clusters]

# Create plot with clusters
fig = px.scatter(
    df_umap, 
    x='x', 
    y='y', 
    color='cluster',
    symbol='category',
    hover_data={'paper': True, 'x': False, 'y': False, 'category': True, 'cluster': True},
    title=f'Automatic Clustering of Research Papers (K={n_clusters})',
    width=1000,
    height=700,
    color_discrete_sequence=px.colors.qualitative.Pastel
)

fig.update_traces(
    marker=dict(size=16, line=dict(width=2, color='white')),
)

fig.update_layout(
    xaxis_title='UMAP Dimension 1',
    yaxis_title='UMAP Dimension 2',
    legend_title='Detected Clusters',
    hovermode='closest'
)

fig.show()

# Show what's in each cluster
print("\n📊 Cluster Contents:")
print("="*80)
for i in range(n_clusters):
    cluster_papers = [papers[j] for j in range(len(papers)) if clusters[j] == i]
    print(f"\n🔹 Cluster {i+1} ({len(cluster_papers)} papers):")
    for paper in cluster_papers:
        print(f"   • {paper}")

## 🎓 Exercises for Students

Now it's your turn! Try these exercises to deepen your understanding.

### Exercise 1: Build Your Own Document Search Engine

Create a collection of at least 10 documents (could be movie summaries, book descriptions, news articles, etc.) and implement semantic search.

**Tasks:**
1. Create your own list of documents
2. Generate embeddings
3. Test with 3 different search queries
4. Analyze the results - did it find the right documents?

In [ ]:
# Exercise 1: Your code here

# Example structure:
my_documents = [
    # Add your documents here
]

# Generate embeddings
# my_embeddings = ...

# Test searches
# query1 = "..."
# results = semantic_search(...)


### Exercise 2: Semantic Analogies Challenge

Test the embedding model with different types of analogies:

1. **Capital cities**: "Tokyo is to Japan as Berlin is to ?"
2. **Professions**: "Nurse is to hospital as chef is to ?"
3. **Animals**: "Cub is to bear as calf is to ?"
4. **Technology**: "iOS is to Apple as Android is to ?"

Create at least 3 of your own analogies and test them!

In [ ]:
# Exercise 2: Your code here

# Use the semantic_analogy function from earlier
# Example:
# semantic_analogy('Tokyo', 'Japan', 'Berlin', 
#                 ['Germany', 'France', 'Italy', 'Spain', 'Poland'])


### Exercise 3: Topic Clustering

Collect sentences or short paragraphs from 3-4 different topics (e.g., sports, politics, technology, entertainment).

**Tasks:**
1. Create a dataset with at least 12 texts (3 per topic)
2. Generate embeddings
3. Use K-means clustering to automatically group them
4. Visualize with UMAP or t-SNE
5. Check if the algorithm correctly identified the topics

In [ ]:
# Exercise 3: Your code here

my_texts = [
    # Topic 1: Sports
    # ...
    
    # Topic 2: Politics
    # ...
    
    # Topic 3: Technology
    # ...
    
    # Topic 4: Entertainment
    # ...
]

# Your clustering and visualization code here


### Exercise 4: Similarity Threshold Experiment

**Question:** What's a good similarity threshold for determining if two documents are "similar"?

**Tasks:**
1. Create pairs of sentences with varying similarity levels:
   - Very similar (paraphrases)
   - Somewhat similar (related topics)
   - Not similar (different topics)
2. Calculate their cosine similarities
3. Determine a good threshold value (e.g., > 0.7 = similar, 0.4-0.7 = related, < 0.4 = different)
4. Visualize the distribution of similarity scores

In [ ]:
# Exercise 4: Your code here

sentence_pairs = [
    # Very similar pairs
    ("sentence 1", "sentence 2"),
    
    # Somewhat similar pairs
    # ...
    
    # Not similar pairs
    # ...
]

# Calculate similarities and create visualization


### Bonus Exercise: Multi-language Embeddings

Try using a multilingual model like `paraphrase-multilingual-MiniLM-L12-v2` to:
1. Embed sentences in different languages
2. Find similar sentences across languages
3. Test if "Hello" in English is close to "Hola" in Spanish, "Bonjour" in French, etc.

In [ ]:
# Bonus Exercise: Your code here

# Load multilingual model
# multilingual_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# Test cross-lingual similarity


## 🎯 Key Takeaways

### What we learned:

1. **Embeddings** convert text into numerical vectors that capture semantic meaning
2. **Cosine similarity** measures how similar two embeddings are (ranging from -1 to 1)
3. **Vector arithmetic** allows us to perform analogies and reasoning (e.g., king - man + woman ≈ queen)
4. **Semantic search** finds similar documents based on meaning, not just keywords
5. **Dimensionality reduction** (t-SNE, UMAP) helps us visualize high-dimensional embeddings in 2D
6. **Clustering** can automatically group similar documents together

### Real-world applications:

- 🔍 **Search engines**: Google, Bing use embeddings for better search results
- 🎬 **Recommendation systems**: Netflix, Spotify, YouTube use embeddings to recommend content
- 💬 **Chatbots & virtual assistants**: Understanding user intent and finding relevant responses
- 📄 **Document classification**: Automatically categorizing emails, support tickets, news articles
- 🌐 **Translation & multilingual search**: Finding similar content across languages
- 🧬 **Scientific research**: Finding related papers, analyzing research trends

### Next steps:

- Explore different embedding models (BERT, RoBERTa, GPT-based models)
- Learn about fine-tuning embeddings for specific domains
- Build a full semantic search application
- Explore vector databases (Pinecone, Weaviate, Milvus) for production use

---

**Great job! 🎉 You now understand the fundamentals of embeddings and semantic similarity!**